# NN eksperimenti — registry-based pipeline

Umjesto da svaki model ima svoju ćeliju sa cijelim training/eval kodom (kao u
`classic_models_cnn.ipynb`/`classic_models_cnn_fast.ipynb`), modeli se ovdje **registruju** u
`nn_utils.py` (`MODEL_REGISTRY`) i pozivaju po imenu preko `run_experiment(...)`.

Svaki poziv `run_experiment` sam radi train/val split, trening, work-level evaluaciju (soft-vote), i
**čuva sve** pod `output/{model_name}_{hash}/`:
- `config.json`, `metrics.json`, `history.json`
- `model.pt` (težine sa najboljim val_loss-om, ne zadnja epoha)
- `plots/loss_curve.png`, `plots/confusion_matrix.png`
- `report.pdf`

`hash` je izveden iz cijele konfiguracije (arhitektura + trening hiperparametri) — ista konfiguracija
uvijek ide u isti folder, drugačija dobija nov folder. Ništa se ne prepisuje slučajno.

In [2]:
%load_ext autoreload
%autoreload 2

# Ucitavanje podataka

## Učitavanje i pregled metapodataka

`metadata` je tabela u kojoj svaki red opisuje jedan audio zapis / stav: kompozitora, djelo, stav, ansambl,
trajanje itd. Naredne ćelije prvo provjeravaju nedostajuće vrijednosti i spisak kompozitora, a tek onda
odlučuju koje klase ostaju u eksperimentu.

In [3]:
import numpy as np
import pandas as pd

from paths import AUDIO_DIR, METADATA_PATH

metadata = pd.read_csv(METADATA_PATH)

print(metadata.head())
print(metadata.columns)

     id  composer               composition                   movement  \
0  1727  Schubert  Piano Quintet in A major                 2. Andante   
1  1728  Schubert  Piano Quintet in A major         3. Scherzo: Presto   
2  1729  Schubert  Piano Quintet in A major  4. Andantino - Allegretto   
3  1730  Schubert  Piano Quintet in A major          5. Allegro giusto   
4  1733  Schubert   Piano Sonata in A major               2. Andantino   

        ensemble            source                      transcriber  \
0  Piano Quintet  European Archive  http://tirolmusic.blogspot.com/   
1  Piano Quintet  European Archive  http://tirolmusic.blogspot.com/   
2  Piano Quintet  European Archive  http://tirolmusic.blogspot.com/   
3  Piano Quintet  European Archive  http://tirolmusic.blogspot.com/   
4     Solo Piano          Museopen                Segundo G. Yogore   

  catalog_name  seconds  
0        OP114      447  
1        OP114      251  
2        OP114      444  
3        OP114      368 

In [4]:
metadata.isna().sum()
# No missing values

id              0
composer        0
composition     0
movement        0
ensemble        0
source          0
transcriber     0
catalog_name    0
seconds         0
dtype: int64

In [5]:
metadata["composer"].unique()
# We have 10 composers

<StringArray>
[ 'Schubert',    'Mozart',    'Dvorak',   'Cambini',     'Haydn',    'Brahms',
     'Faure',     'Ravel',      'Bach', 'Beethoven']
Length: 10, dtype: str

## Zašto se pravi `work_id`?

Jedno muzičko djelo može imati više stavova, a u metapodacima su oni posebni redovi. Kombinacijom
`composer + composition` pravi se identifikator cijelog djela. Tako npr. više stavova iste sonate ostaju ista
grupa tokom cross-validationa.

Nakon toga se broji broj **različitih djela po kompozitoru**, ne broj segmenata. Kompozitori sa manje od 5
djela se uklanjaju jer bi grupni cross-validation za njih bio vrlo nestabilan.


In [6]:
metadata["work_id"] = (
    metadata["composer"].astype(str)
    + " | "
    + metadata["composition"].astype(str)
)

# Adding a column so we can group the same composition from different movements

In [7]:
work_counts = (
    metadata.groupby("composer")["work_id"]
    .nunique()
)

print(work_counts)

# We will remove all the composers that have less than 5 compositions in the dataset, also we can see our dataset is inbalanced

composer
Bach         30
Beethoven    55
Brahms        8
Cambini       3
Dvorak        2
Faure         1
Haydn         1
Mozart       11
Ravel         1
Schubert      9
Name: work_id, dtype: int64


In [8]:
valid_composers = work_counts[
    work_counts >= 5
].index

metadata_filtered = metadata[
    metadata["composer"].isin(valid_composers)
].copy()

# Filtering the composers

In [9]:
print(
    metadata_filtered.groupby("composer")["work_id"]
    .nunique()
    .sort_values()
)

composer
Brahms        8
Schubert      9
Mozart       11
Bach         30
Beethoven    55
Name: work_id, dtype: int64


---

## Mel-spektrogram dataset


In [ ]:
from spectrograms import make_spectrogram_dataset

X, y, groups = make_spectrogram_dataset(metadata_filtered, AUDIO_DIR)

print(X.shape)

---

## Pokretanje registrovanih modela

`cnn_small` i `crnn_small` su trenutno registrovani u `nn_utils.py` (`MODEL_REGISTRY`) — oba manji od
`SimpleCNN` korišćenog u ranijim notebook-ima, sa opcionim (Bi)GRU slojem nad vremenskom osom prije
poolinga. Svaki poziv niže je nezavisan eksperiment — parametri koji nisu navedeni uzimaju default iz
registry-ja.

In [ ]:
from nn_utils import run_experiment

result_cnn = run_experiment(
    "cnn_small", X, y, groups,
    epochs=20, batch_size=64, dropout=0.6
)

In [ ]:
result_crnn = run_experiment(
    "crnn_small", X, y, groups,
    epochs=20, batch_size=64, gru_hidden=32, dropout=0.6
)

---

## Poređenje

Rezultati su i printovani iznad i sačuvani na disku — ovo je samo brz pregled u samom notebook-u.

In [ ]:
import pandas as pd

pd.DataFrame([
    {"model": "cnn_small", "accuracy": result_cnn["accuracy"], "macro_f1": result_cnn["macro_f1"], "run_dir": result_cnn["run_dir"]},
    {"model": "crnn_small", "accuracy": result_crnn["accuracy"], "macro_f1": result_crnn["macro_f1"], "run_dir": result_crnn["run_dir"]},
])

---

## RF + CNN ensemble

Kombinuje `cnn_small` (na spektrogramima) i Random Forest (na ručnim audio atributima iz
`feature_engineering.py`) — usrednjavanjem njihovih work-level vjerovatnoća. `run_ensemble_experiment`
sam obezbjeđuje da oba modela vide **identičan skup djela** u train/val split-u (`split_by_work`), iako
imaju različit broj segmenata po djelu (25s audio segmenti vs 10s spektrogram segmenti).

In [ ]:
from dataset import make_dataset

X_audio, y_audio, groups_audio = make_dataset(metadata_filtered, AUDIO_DIR)

print(X_audio.shape)

In [ ]:
from nn_utils import run_ensemble_experiment

result_ensemble = run_ensemble_experiment(
    "cnn_small", X, y, groups, X_audio, y_audio, groups_audio,
    epochs=20, batch_size=64, dropout=0.6
)

---

## PANNs transfer learning


`make_panns_dataset` (audio se ponovo dekodira na 32kHz — PANNs očekuje tu stopu, ne 44.1kHz kao ostatak
projekta, pa se ne može iskoristiti postojeći keš) računa 2048-dim embedding po segmentu preko pretreniranog
Cnn14 modela (treniran na AudioSet-u, milioni zvučnih isječaka). `run_panns_experiment` onda trenira samo
mali RF klasifikator na vrhu tih (zamrznutih) embedinga — to je "transfer learning" dio: ne treniramo
enkoder od nule, samo iskorišćavamo već naučenu reprezentaciju.

In [11]:
import time
from panns_features import make_panns_dataset

# small subset first - confirms the checkpoint download works and gives a real per-recording timing
# before committing to the full 302-recording dataset
_sample = metadata_filtered.head(10)

_start = time.time()
_Xs, _ys, _gs = make_panns_dataset(_sample, AUDIO_DIR, use_cache=False)
_elapsed = time.time() - _start

print(f"{len(_sample)} recordings -> {_Xs.shape} in {_elapsed:.1f}s ({_elapsed/len(_sample):.1f}s/recording)")
print(f"estimated full dataset (302 recordings): ~{_elapsed/len(_sample)*302/60:.1f} min")

Processing PANNs embeddings:   0%|          | 0/10 [00:00<?, ?it/s]

Checkpoint path: C:\Users\marko.vucic_ominimo/panns_data/Cnn14_mAP=0.431.pth
Using CPU.


Processing PANNs embeddings:  10%|█         | 1/10 [01:06<09:54, 66.02s/it]


KeyboardInterrupt: 

In [ ]:
X_panns, y_panns, groups_panns = make_panns_dataset(metadata_filtered, AUDIO_DIR)

print(X_panns.shape)

In [ ]:
from nn_utils import run_panns_experiment

result_panns = run_panns_experiment(X_panns, y_panns, groups_panns)

---

## Puno poređenje

In [ ]:
rows = []
for name, r in [("cnn_small", result_cnn), ("crnn_small", result_crnn),
                ("ensemble_rf_cnn", result_ensemble), ("panns_rf", result_panns)]:
    if r.get("status") == "ok":
        rows.append({"model": name, "accuracy": r["accuracy"], "macro_f1": r["macro_f1"], "run_dir": r["run_dir"]})
    else:
        rows.append({"model": name, "accuracy": None, "macro_f1": None, "run_dir": r["run_dir"], "status": r.get("status")})

pd.DataFrame(rows)